In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the "../input/" directory.
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
train_data = pd.read_csv("/kaggle/input/titanic/train.csv")
train_data.head()

In [ ]:
test_data = pd.read_csv("/kaggle/input/titanic/test.csv")
test_data.head()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

# mdelva
train_df = pd.read_csv("/kaggle/input/titanic/train.csv")
test_df = pd.read_csv("/kaggle/input/titanic/test.csv")

# Simpan PassengerId untuk file submisi nanti
test_passenger_ids = test_df["PassengerId"]

# --- 2. Visualisasi (Analisis Data Eksploratif) ---
# Menggunakan ide dari Skrip 1
plt.figure(figsize=(8, 6))
sns.countplot(x='Survived', data=train_df, palette='pastel')
plt.title('Distribusi Penumpang Selamat (1) vs. Meninggal (0)', fontsize=14)
plt.xlabel('Status Bertahan Hidup', fontsize=12)
plt.ylabel('Jumlah Penumpang', fontsize=12)
plt.xticks(ticks=[0, 1], labels=['Meninggal (0)', 'Selamat (1)'])
plt.savefig('survival_distribution.png')
print("Grafik 'survival_distribution.png' berhasil dibuat.")

# --- 3. Preprocessing & Feature Engineering ---

# Pisahkan target (y) dari data training
y_train = train_df["Survived"]
# Hapus 'Survived' dari train_df sebelum digabung
train_df = train_df.drop("Survived", axis=1)

# Gabungkan data train dan test untuk preprocessing yang konsisten (Ide dari Skrip 1)
combined_df = pd.concat([train_df, test_df], ignore_index=True)

# --- Menangani Nilai yang Hilang (Missing Values) ---
# Isi 'Age' dan 'Fare' dengan MEDIAN (lebih baik dari mean untuk data miring)
combined_df['Age'] = combined_df['Age'].fillna(combined_df['Age'].median())
combined_df['Fare'] = combined_df['Fare'].fillna(combined_df['Fare'].median())
# Isi 'Embarked' dengan MODUS (nilai paling sering muncul)
combined_df['Embarked'] = combined_df['Embarked'].fillna(combined_df['Embarked'].mode()[0])

# --- Feature Engineering (Perbaikan) ---
# Buat fitur 'FamilySize' dari 'SibSp' dan 'Parch'
combined_df['FamilySize'] = combined_df['SibSp'] + combined_df['Parch'] + 1
# Buat fitur 'IsAlone'
combined_df['IsAlone'] = 0
combined_df.loc[combined_df['FamilySize'] == 1, 'IsAlone'] = 1

# --- Mengubah Fitur Kategorikal ---
# Ubah 'Sex' menjadi biner (0 atau 1)
combined_df['Sex'] = combined_df['Sex'].map({'male': 0, 'female': 1})
# Ubah 'Embarked' menjadi dummy variables (Ide dari Skrip 1)
combined_df = pd.get_dummies(combined_df, columns=['Embarked'], drop_first=True)

# --- Feature Selection (Hapus kolom yang tidak perlu) ---
# Hapus kolom asli yang sudah tidak relevan atau tidak informatif
drop_cols = ['Name', 'Ticket', 'Cabin', 'PassengerId', 'SibSp', 'Parch']
combined_df = combined_df.drop(columns=drop_cols)

# --- Feature Scaling (Perbaikan) ---
# Terapkan StandardScaler ke fitur numerik agar skalanya seragam
# Ini sangat penting untuk banyak model, dan praktik yang baik untuk semua.
scaler = StandardScaler()
# Tentukan kolom mana yang perlu di-scale (bukan biner/dummy)
cols_to_scale = ['Age', 'Fare', 'FamilySize', 'Pclass']
combined_df[cols_to_scale] = scaler.fit_transform(combined_df[cols_to_scale])

# --- 4. Memisahkan Kembali Data Train dan Test ---
# Pisahkan kembali data setelah semua preprocessing selesai
X_train = combined_df.iloc[:len(train_df)]
X_test = combined_df.iloc[len(train_df):]

print(f"Bentuk data training (fitur): {X_train.shape}")
print(f"Bentuk data test (fitur): {X_test.shape}")

# 5) latih model
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model.fit(X_train, y_train)

# 6) prediksi/file submisi
predictions = model.predict(X_test)

# dataframe buat submisi
submission_df = pd.DataFrame({
    "PassengerId": test_passenger_ids,
    "Survived": predictions
})

# simpan ke file CSV buat di submit ke kaggle
submission_df.to_csv("submission.csv", index=False)

print("File 'submission.csv' berhasil dibuat!")
print(submission_df.head())